# RL (PPO) no Portfólio — Latent Fusion

Treina PPO com `TradingEnv` sobre o mesmo portfólio de 49 ações US do `final_report.ipynb`.
Compara 4 presets de reward (aggressive, balanced, vol_focused, conservative) contra buy & hold.

In [1]:
import sys, os, warnings
from pathlib import Path
_root = Path(os.getcwd())
while not ((_root / 'src').exists() and (_root / 'pyproject.toml').exists()) and _root != _root.parent:
    _root = _root.parent
sys.path.insert(0, str(_root))
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from src.backtest.engine import BacktestEngine, BacktestConfig
from src.backtest.visualization import BG, PANEL, GOLD, GREEN, RED, WHITE, CYAN, PURPLE, ORANGE
from src.models.rl_env import TradingEnv, REWARD_PRESETS, make_env, train_ppo
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 160)
_images = _root / 'images'
_images.mkdir(exist_ok=True)

## 1. Dados — mesmo portfólio do final_report

In [2]:
cached = pd.read_pickle(_root / 'cache/data/top50_prepared_data.pkl')
ret_df = cached['ret_df']
mask = cached['mask']
split = 2694

bh_w = mask.div(mask.sum(axis=1).replace(0.0, float('nan')), axis=0).fillna(0.0)
bh_ret = (bh_w * ret_df.fillna(0.0)).sum(axis=1)
price_port = (1 + bh_ret).cumprod() * 100.0

rng = np.random.default_rng(42)
df_port = pd.DataFrame({
    'timestamp': ret_df.index,
    'close': price_port.values,
    'open': price_port.values * (1 + rng.standard_normal(len(price_port)) * 0.001),
    'high': price_port.values * (1 + np.abs(rng.standard_normal(len(price_port))) * 0.002),
    'low': price_port.values * (1 - np.abs(rng.standard_normal(len(price_port))) * 0.002),
    'volume': np.ones(len(price_port)) * 1e6,
})

df_train = df_port.iloc[:split].copy()
df_test = df_port.iloc[split:].copy()

config = BacktestConfig(initial_cash=10_000, fee_bps=4, slippage_bps=1, periods_per_year=252)

pd.DataFrame({
    'split': ['train', 'test'],
    'days': [len(df_train), len(df_test)],
    'from': [str(df_train['timestamp'].iloc[0].date()), str(df_test['timestamp'].iloc[0].date())],
    'to': [str(df_train['timestamp'].iloc[-1].date()), str(df_test['timestamp'].iloc[-1].date())],
    'bh_return_pct': [
        (df_train['close'].iloc[-1]/df_train['close'].iloc[0]-1)*100,
        (df_test['close'].iloc[-1]/df_test['close'].iloc[0]-1)*100,
    ]
})

,split,days,from,to,bh_return_pct
0,train,2694,2009-04-14,2020-07-08,319.138723
1,test,1155,2020-07-09,2025-03-27,91.335798


## 2. Treinar PPO — 4 Reward Presets

Cada preset treina 50k timesteps no conjunto de treino. Avalia no teste.

In [3]:
TIMESTEPS = 50_000
results = {}

for preset_name in REWARD_PRESETS.keys():
    env_train = make_env(df_train, preset=preset_name, lookback=20)
    vec_env = DummyVecEnv([lambda: env_train])
    model = PPO('MlpPolicy', vec_env, learning_rate=3e-4, verbose=0, device='auto',
                n_steps=2048, batch_size=256, gamma=0.99, ent_coef=0.01)
    model.learn(total_timesteps=TIMESTEPS, progress_bar=False)

    env_test = make_env(df_test, preset=preset_name, lookback=20)
    obs, _ = env_test.reset()
    done = False
    portfolio_values = [env_test.initial_balance]
    actions_taken = []
    rewards = []
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, done, trunc, info = env_test.step(int(action))
        portfolio_values.append(info['portfolio_value'])
        actions_taken.append(int(action))
        rewards.append(reward)
        done = done or trunc

    pv = np.array(portfolio_values)
    total_ret = (pv[-1] / pv[0] - 1) * 100
    bh_ret = (df_test['close'].iloc[-1] / df_test['close'].iloc[0] - 1) * 100
    daily_rets = np.diff(pv) / pv[:-1]
    sharpe = (np.mean(daily_rets) / (np.std(daily_rets) + 1e-8)) * np.sqrt(252)
    cum = np.cumprod(1 + daily_rets)
    dd = (cum / np.maximum.accumulate(cum) - 1) * 100
    max_dd = float(dd.min())
    n_buys = sum(1 for a in actions_taken if a == 1)
    n_sells = sum(1 for a in actions_taken if a == 2)
    n_holds = sum(1 for a in actions_taken if a == 0)

    results[preset_name] = {
        'return_pct': round(total_ret, 2),
        'bh_return_pct': round(bh_ret, 2),
        'excess_pct': round(total_ret - bh_ret, 2),
        'sharpe': round(sharpe, 3),
        'max_dd_pct': round(max_dd, 1),
        'n_buys': n_buys,
        'n_sells': n_sells,
        'n_holds': n_holds,
        'final_portfolio': round(pv[-1], 2),
        'portfolio_values': pv,
        'actions': actions_taken,
        'rewards': rewards,
        'model': model,
    }

res_df = pd.DataFrame([{k: v for k, v in r.items() if k not in ('portfolio_values','actions','rewards','model')} for r in results.values()])
res_df.index = list(results.keys())
res_df

,return_pct,bh_return_pct,excess_pct,sharpe,max_dd_pct,n_buys,n_sells,n_holds,final_portfolio
aggressive,0.00,91.34,-91.34,0.000,0.0,0,8,1146,10000.00
balanced,91.14,91.34,-0.19,0.981,-19.3,1,0,1153,19114.47
vol_focused,91.14,91.34,-0.19,0.981,-19.3,1154,0,0,19114.47
conservative,-32.57,91.34,-123.91,-0.699,-33.3,448,81,625,6742.86


## 3. Equity Curves — PPO vs Buy & Hold

In [4]:
fig, ax = plt.subplots(figsize=(22, 11))
fig.patch.set_facecolor(BG)

test_dates = df_test['timestamp'].values
bh_curve = df_test['close'].values / df_test['close'].iloc[0] * 10_000
ax.plot(test_dates, bh_curve, color=WHITE, lw=1.5, ls='--', alpha=0.5, label=f'Buy & Hold ({res_df.loc[res_df.index[0], "bh_return_pct"]:+.1f}%)')

colors = [GOLD, CYAN, PURPLE, ORANGE]
for i, (name, r) in enumerate(results.items()):
    pv = r['portfolio_values']
    ax.plot(test_dates[:len(pv)], pv, color=colors[i], lw=2, label=f'PPO {name} ({r["return_pct"]:+.1f}%)')

ax.axhline(10_000, color='white', ls='--', alpha=0.15)
ax.set_title('PPO Agent — 4 Reward Presets vs Buy & Hold (49 US Stocks, Test Period)', color='white', fontsize=14, fontweight='bold')
ax.set_facecolor(PANEL)
ax.tick_params(colors='white', labelsize=13)
ax.grid(True, alpha=0.12)
ax.legend(fontsize=14, facecolor=PANEL, edgecolor='white', labelcolor='white', loc='upper left')
fig.tight_layout()
fig.savefig(_images / 'rl_ppo_equity_curves.png', dpi=300, facecolor=BG, edgecolor='none', bbox_inches='tight')
plt.show()

## 4. Action Distribution por Preset

In [5]:
fig, ax = plt.subplots(figsize=(22, 8))
fig.patch.set_facecolor(BG)

presets = list(results.keys())
x = np.arange(len(presets))
w = 0.25
buys = [results[p]['n_buys'] for p in presets]
sells = [results[p]['n_sells'] for p in presets]
holds = [results[p]['n_holds'] for p in presets]

ax.bar(x - w, buys, w, color=GREEN, alpha=0.85, label='BUY')
ax.bar(x, sells, w, color=RED, alpha=0.85, label='SELL')
ax.bar(x + w, holds, w, color=WHITE, alpha=0.4, label='HOLD')
ax.set_xticks(x)
ax.set_xticklabels(presets, color='white', fontsize=14)
ax.set_title('PPO Action Distribution by Reward Preset', color='white', fontsize=14, fontweight='bold')
ax.set_facecolor(PANEL)
ax.tick_params(colors='white', labelsize=13)
ax.grid(True, alpha=0.12, axis='y')
ax.legend(fontsize=14, facecolor=PANEL, edgecolor='white', labelcolor='white')
fig.tight_layout()
fig.savefig(_images / 'rl_ppo_action_distribution.png', dpi=300, facecolor=BG, edgecolor='none', bbox_inches='tight')
plt.show()

## 5. Métricas Comparativas — PPO vs Estratégias Determinísticas

In [6]:
from src.strategy import SmaCrossStrategy, MeanReversionStrategy, RegimeRouterStrategy
engine = BacktestEngine(config)

det_strats = {
    'SMA 20/100': SmaCrossStrategy(20, 100),
    'SMA 10/50': SmaCrossStrategy(10, 50),
    'Mean Reversion': MeanReversionStrategy(lookback=50, z_entry=1.5, z_exit=0.3),
    'Regime Router': RegimeRouterStrategy(vol_window=20, regime_percentile=50, mom_window=5),
}

all_rows = []
for name, strat in det_strats.items():
    r = engine.run(df_test, strat)
    m = r.metrics
    all_rows.append({
        'strategy': name, 'type': 'deterministic',
        'return_pct': round(m['total_return_pct'], 2),
        'bh_return_pct': round(m['bh_return_pct'], 2),
        'excess_pct': round(m['excess_return_pct'], 2),
        'sharpe': round(m['sharpe'], 3),
        'max_dd_pct': round(m['max_drawdown_pct'], 1),
    })

for name, r in results.items():
    all_rows.append({
        'strategy': f'PPO {name}', 'type': 'RL',
        'return_pct': r['return_pct'], 'bh_return_pct': r['bh_return_pct'],
        'excess_pct': r['excess_pct'], 'sharpe': r['sharpe'],
        'max_dd_pct': r['max_dd_pct'],
    })

all_rows.append({
    'strategy': 'Buy & Hold', 'type': 'baseline',
    'return_pct': round((df_test['close'].iloc[-1]/df_test['close'].iloc[0]-1)*100, 2),
    'bh_return_pct': 0, 'excess_pct': 0, 'sharpe': 0, 'max_dd_pct': 0,
})

comp = pd.DataFrame(all_rows).sort_values('excess_pct', ascending=False).reset_index(drop=True)
comp

,strategy,type,return_pct,bh_return_pct,excess_pct,sharpe,max_dd_pct
0,Buy & Hold,baseline,91.34,0.00,0.00,0.000,0.0
1,PPO balanced,RL,91.14,91.34,-0.19,0.981,-19.3
2,PPO vol_focused,RL,91.14,91.34,-0.19,0.981,-19.3
3,SMA 20/100,deterministic,19.64,91.34,-71.70,0.397,-22.9
4,Mean Reversion,deterministic,6.27,91.34,-85.06,0.170,-22.4
5,SMA 10/50,deterministic,4.90,91.34,-86.44,0.148,-28.5
6,PPO aggressive,RL,0.00,91.34,-91.34,0.000,0.0
7,Regime Router,deterministic,-29.26,91.34,-120.60,-0.410,-39.0
8,PPO conservative,RL,-32.57,91.34,-123.91,-0.699,-33.3


## 6. Excess Return — Bar Chart

In [7]:
fig, ax = plt.subplots(figsize=(22, 9))
fig.patch.set_facecolor(BG)

plot_df = comp[comp['strategy'] != 'Buy & Hold'].copy()
y_pos = np.arange(len(plot_df))
excess = plot_df['excess_pct'].values
colors = [GREEN if v > 0 else RED for v in excess]

ax.barh(y_pos, excess, color=colors, alpha=0.85, edgecolor='white', lw=0.3)
ax.set_yticks(y_pos)
ax.set_yticklabels(plot_df['strategy'], color='white', fontsize=13)
ax.axvline(0, color='white', lw=0.5)
for i, v in enumerate(excess):
    ax.text(v + (0.5 if v >= 0 else -3), i, f'{v:+.1f}%', va='center', color='white', fontsize=12)
ax.set_title('Excess Return vs Buy & Hold — RL (PPO) vs Deterministic Strategies', color='white', fontsize=14, fontweight='bold')
ax.set_facecolor(PANEL)
ax.tick_params(colors='white', labelsize=13)
ax.grid(True, alpha=0.12, axis='x')
fig.tight_layout()
fig.savefig(_images / 'rl_ppo_excess_comparison.png', dpi=300, facecolor=BG, edgecolor='none', bbox_inches='tight')
plt.show()

## 7. Reward Curve durante Treino

In [8]:
fig, ax = plt.subplots(figsize=(22, 8))
fig.patch.set_facecolor(BG)

colors = [GOLD, CYAN, PURPLE, ORANGE]
for i, (name, r) in enumerate(results.items()):
    cum_reward = np.cumsum(r['rewards'])
    ax.plot(cum_reward, color=colors[i], lw=1.5, label=f'{name} (total={cum_reward[-1]:.4f})')

ax.set_title('Cumulative Reward During Test Episode — PPO Presets', color='white', fontsize=14, fontweight='bold')
ax.set_facecolor(PANEL)
ax.tick_params(colors='white', labelsize=13)
ax.grid(True, alpha=0.12)
ax.legend(fontsize=14, facecolor=PANEL, edgecolor='white', labelcolor='white')
fig.tight_layout()
fig.savefig(_images / 'rl_ppo_reward_curves.png', dpi=300, facecolor=BG, edgecolor='none', bbox_inches='tight')
plt.show()

## 8. Sharpe vs Max Drawdown — Scatter

In [9]:
fig, ax = plt.subplots(figsize=(14, 10))
fig.patch.set_facecolor(BG)

for _, row in comp.iterrows():
    if row['strategy'] == 'Buy & Hold':
        c, marker, sz = WHITE, 's', 150
    elif row['type'] == 'RL':
        c, marker, sz = GOLD, '^', 200
    else:
        c, marker, sz = CYAN, 'o', 120
    ax.scatter(abs(row['max_dd_pct']), row['sharpe'], c=c, s=sz, marker=marker, zorder=5, edgecolors='white', lw=0.5)
    ax.annotate(row['strategy'], (abs(row['max_dd_pct']), row['sharpe']),
                textcoords='offset points', xytext=(8, 5), color='white', fontsize=11)

ax.set_xlabel('Max Drawdown (%)', color='white', fontsize=14)
ax.set_ylabel('Sharpe Ratio', color='white', fontsize=14)
ax.set_title('Risk-Return Map — RL vs Deterministic vs BH', color='white', fontsize=14, fontweight='bold')
ax.set_facecolor(PANEL)
ax.tick_params(colors='white', labelsize=13)
ax.grid(True, alpha=0.12)
fig.tight_layout()
fig.savefig(_images / 'rl_ppo_risk_return_map.png', dpi=300, facecolor=BG, edgecolor='none', bbox_inches='tight')
plt.show()

## 9. Resumo Final

In [10]:
comp[['strategy', 'type', 'return_pct', 'bh_return_pct', 'excess_pct', 'sharpe', 'max_dd_pct']]

,strategy,type,return_pct,bh_return_pct,excess_pct,sharpe,max_dd_pct
0,Buy & Hold,baseline,91.34,0.00,0.00,0.000,0.0
1,PPO balanced,RL,91.14,91.34,-0.19,0.981,-19.3
2,PPO vol_focused,RL,91.14,91.34,-0.19,0.981,-19.3
3,SMA 20/100,deterministic,19.64,91.34,-71.70,0.397,-22.9
4,Mean Reversion,deterministic,6.27,91.34,-85.06,0.170,-22.4
5,SMA 10/50,deterministic,4.90,91.34,-86.44,0.148,-28.5
6,PPO aggressive,RL,0.00,91.34,-91.34,0.000,0.0
7,Regime Router,deterministic,-29.26,91.34,-120.60,-0.410,-39.0
8,PPO conservative,RL,-32.57,91.34,-123.91,-0.699,-33.3
